In [1]:
import requests
import datetime
from tqdm import tqdm

In [2]:
# Replace with your actual API token from Travelpayouts 
# You can just google "aviaseils API", easy to register and all is free, 
# but this data is some kind of cache, so it is not too representative, but still good enough 
TOKEN = "enter1here1smth1rbrbbrbrbbrbrbrb3242brbrb"

# Construct the URL with the desired parameters:
# - origin: Moscow (MOW)
# - destination: Tokyo (TYO)
# - departure date: 2025-03-28
# - return date: 2025-04-10
# - one_way=false for round-trip offers
# - sorting: Sorting by price, 
# - direct=false: including non-direct flights, 
# - cy: currency in USD (RUB)
# - limit, page: how many tickets will be loaded from pages (list sorted by sorting param) 
url = (
    "https://api.travelpayouts.com/aviasales/v3/prices_for_dates?"
    "origin=MOW&destination=TYO&departure_at=2025-03-28&return_at=2025-04-10"
    "&unique=false&sorting=price&direct=false&cy=RUB&limit=30&page=1&one_way=false"
    f"&token={TOKEN}"
)


Test check of API.

In [3]:
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    if data.get("success") and data.get("data"):
        flights = data["data"]
        print("\n✈️  Cheapest Flights from Moscow (MOW) to Tokyo (TYO):\n")
        for idx, flight in enumerate(flights, 1):
            price = flight.get("price", "N/A")
            airline = flight.get("airline", "N/A")
            flight_number = flight.get("flight_number", "N/A")
            departure_at = flight.get("departure_at", "N/A")
            return_at = flight.get("return_at", "N/A")
            # Calculate total transfers (for both outbound and return legs)
            transfers = flight.get("transfers", 0)
            return_transfers = flight.get("return_transfers", 0)
            total_transfers = transfers + return_transfers
            link = flight.get("link", "")
            full_link = f"https://www.aviasales.com{link}" if link else "N/A"

            print(f"🔹 Option {idx}:")
            print(f"   - 💰 Price: ${price}")
            print(f"   - ✈️ Airline: {airline} (Flight No: {flight_number})")
            print(f"   - 🛫 Departure: {departure_at}")
            print(f"   - 🛬 Return: {return_at}")
            print(f"   - ⏳ Total Transfers: {total_transfers}")
            print(f"   - 🔗 Ticket Link: {full_link}\n")
    else:
        print("❌ No flight data found for the specified dates and parameters.")
else:
    print(f"❌ Error {response.status_code}: {response.text}")



✈️  Cheapest Flights from Moscow (MOW) to Tokyo (TYO):

🔹 Option 1:
   - 💰 Price: $78035
   - ✈️ Airline: MU (Flight No: 2076)
   - 🛫 Departure: 2025-03-17T17:05:00+03:00
   - 🛬 Return: 2025-03-30T17:55:00+09:00
   - ⏳ Total Transfers: 4
   - 🔗 Ticket Link: https://www.aviasales.com/search/MOW1703TYO30031?t=J217422311001742374500002030SVOPKXKIXNRT17433573001743419400001395NRTPEKGYDDME_4072774254cfab5be563b5b0c607c1c1_77504&search_date=04022025&expected_price_uuid=78ce8a12-146f-4fd8-9ec2-5e00424bc502&expected_price_source=share&expected_price_currency=rub&expected_price=78035

🔹 Option 2:
   - 💰 Price: $78280
   - ✈️ Airline: CZ (Flight No: 8372)
   - 🛫 Departure: 2025-03-17T21:00:00+03:00
   - 🛬 Return: 2025-03-30T17:55:00+09:00
   - ⏳ Total Transfers: 4
   - 🔗 Ticket Link: https://www.aviasales.com/search/MOW1703TYO30031?t=CZ17422452001742396100002155SVOSZXHRBNRT17433573001743419400001395NRTPEKGYDDME_9b3e394f388d285394ed9caf4afcb6dc_77747&search_date=04022025&expected_price_uuid=78ce

# Main search of tickets

Both sides tickets (3 cities, 12-17 days that should overlap 2 days from date window (2025, 3, 28)-(2025, 4, 9))

In [ ]:
# Search parameters
origin = "MOW"
destinations = ["TYO", "NGO", "OSA"]  # Tokyo, Nagoya, Osaka (IATA codes)
min_duration = 12
max_duration = 17

# The window that must overlap with the trip (at least 2 days)
window_start = datetime.date(2025, 3, 28)
window_end = datetime.date(2025, 4, 9)
window_end_plus = window_end + datetime.timedelta(days=1)  # for inclusive overlap

# Range of possible departure dates (from 2025-03-15 to 2025-04-07)
start_departure = datetime.date(2025, 3, 15)
end_departure = datetime.date(2025, 4, 7)
days_range = (end_departure - start_departure).days + 1

# Pre-calculate all query combinations: (destination, trip_duration, departure_date)
queries = [
    (dest, duration, start_departure + datetime.timedelta(days=offset))
    for dest in destinations
    for duration in range(min_duration, max_duration + 1)
    for offset in range(days_range)
]

results = []  # To store matching flight offers

# Create a session to reuse connections
session = requests.Session()

def build_api_url(origin, destination, departure_date, return_date):
    """Return the API URL for given parameters."""
    d_str = departure_date.strftime("%Y-%m-%d")
    return_str = return_date.strftime("%Y-%m-%d")
    url = (
        "https://api.travelpayouts.com/aviasales/v3/prices_for_dates?"
        f"origin={origin}&destination={destination}&departure_at={d_str}&return_at={return_str}"
        "&unique=false&sorting=price&direct=false&cy=RUB&limit=5&page=1&one_way=false"
        f"&token={TOKEN}"
    )
    return url

# Iterate over all query combinations using a tqdm progress bar.
for destination, trip_duration, departure_date in tqdm(queries, desc="Processing queries"):
    return_date = departure_date + datetime.timedelta(days=trip_duration)
    # Calculate overlap between the trip period and the window
    overlap_start = max(departure_date, window_start)
    overlap_end = min(return_date, window_end_plus)
    overlap_days = (overlap_end - overlap_start).days
    if overlap_days < 2:
        continue

    url = build_api_url(origin, destination, departure_date, return_date)
    try:
        response = session.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get("success") and data.get("data"):
                for flight in data["data"]:
                    price = flight.get("price", 0)
                    try:
                        ntransfers = int(flight.get("transfers", 0) + flight.get("return_transfers", 0))
                    except Exception:
                        ntransfers = 0
                    if price < 65000 and ntransfers < 3:
                        flight_info = {
                            "destination": destination,
                            "departure": departure_date.strftime("%Y-%m-%d"),
                            "return": return_date.strftime("%Y-%m-%d"),
                            "price": price,
                            "airline": flight.get("airline", "N/A"),
                            "flight_number": flight.get("flight_number", "N/A"),
                            "departure_at": flight.get("departure_at", "N/A"),
                            "return_at": flight.get("return_at", "N/A"),
                            "transfers": ntransfers,
                            "link": f"https://www.aviasales.com{flight.get('link', '')}"
                        }
                        results.append(flight_info)
                        # Uncomment the following line to log each matching price:
                        # print(f"Found price: {price}, in {destination}")
        else:
            print(f"Error {response.status_code} for URL: {url}")
    except Exception as e:
        print(f"Request failed: {e}")

# Sort results by price (lowest first)
results.sort(key=lambda x: x["price"])

In [ ]:
# Print the results in a readable format
print("\n✈️  Flights found under 65000 rub:\n")
if results:
    for idx, flight in enumerate(results, 1):
        print(f"🔹 Option {idx}:")
        print(f"   - Destination: {flight['destination']}")
        print(f"   - Price: {flight['price']} rub")
        print(f"   - Trip Dates: {flight['departure']} to {flight['return']}")
        print(f"   - Airline: {flight['airline']} (Flight No: {flight['flight_number']})")
        print(f"   - Departure Time: {flight['departure_at']}")
        print(f"   - Return Time: {flight['return_at']}")
        print(f"   - Total Transfers: {flight['transfers']}")
        print(f"   - Ticket Link: {flight['link']}\n")
else:
    print("No flights found that meet the criteria.")


# Separate tickets 

Two independed tickets but with same pattern (3 cities, 12-17 days that should overlap 2 days from date window (2025, 3, 28)-(2025, 4, 9))

I found for my routs they has poor representation, so its gives worse in price and less results.

In [ ]:
# Search criteria:
min_duration = 12
max_duration = 17
price_threshold = 70000  # Total combined price in USD

# Target window: The trip must overlap this window (28 March–9 April) by at least 2 days.
window_start = datetime.date(2025, 3, 28)
window_end = datetime.date(2025, 4, 9)

# Date range for searches (used for both outbound and inbound flights)
start_date = datetime.date(2025, 3, 15)
end_date = datetime.date(2025, 4, 7)
days_range = (end_date - start_date).days + 1

# Create a session for improved performance
session = requests.Session()

def search_one_way(origin, destination, search_date):
    """
    Search for one-way flights for a given search_date between origin and destination.
    Returns a list of flights that (after conversion) have exactly one transfer.
    """
    flights = []
    date_str = search_date.strftime("%Y-%m-%d")
    url = (
        "https://api.travelpayouts.com/aviasales/v3/prices_for_dates?"
        f"origin={origin}&destination={destination}&departure_at={date_str}"
        "&unique=false&sorting=price&direct=false&cy=RUB&limit=10&page=1&one_way=true"
        f"&token={TOKEN}"
    )
    try:
        response = session.get(url, timeout=10)
        # Uncomment the following line for debugging a specific date:
        # print(f"DEBUG: URL: {url}\nResponse: {response.text}\n")
        if response.status_code == 200:
            data = response.json()
            if data.get("success") and data.get("data"):
                for flight in data["data"]:
                    try:
                        transfers = int(flight.get("transfers", 0))
                    except Exception:
                        transfers = 0
                    # Only accept flights with exactly 1 transfer.
                    if transfers == 1:
                        flights.append({
                            "departure_date": search_date,
                            "price": flight.get("price", 0),
                            "airline": flight.get("airline", "N/A"),
                            "flight_number": flight.get("flight_number", "N/A"),
                            "departure_at": flight.get("departure_at", "N/A"),
                            "link": f"https://www.aviasales.com{flight.get('link', '')}"
                        })
        else:
            print(f"Error {response.status_code} for URL: {url}")
    except Exception as e:
        print(f"Request failed for {url}: {e}")
    return flights

# -------------------------
# Outbound search: MOW -> OSA
# -------------------------
print("Searching outbound flights (MOW -> OSA)...")
outbound_results = []
for day_offset in tqdm(range(days_range), desc="Outbound search"):
    current_date = start_date + datetime.timedelta(days=day_offset)
    outbound_results.extend(search_one_way("MOW", "OSA", current_date))

# -------------------------
# Inbound search: TYO -> MOW
# -------------------------
print("Searching inbound flights (TYO -> MOW)...")
inbound_results = []
for day_offset in tqdm(range(days_range), desc="Inbound search"):
    current_date = start_date + datetime.timedelta(days=day_offset)
    inbound_results.extend(search_one_way("TYO", "MOW", current_date))

# -------------------------
# Combine outbound and inbound flights into round-trip pairs
# -------------------------
combined_results = []
print("Combining outbound and inbound flights...")
for out_flight in tqdm(outbound_results, desc="Combining flights (outbound)"):
    for in_flight in inbound_results:
        # Ensure the inbound flight departs after the outbound flight.
        trip_duration = (in_flight["departure_date"] - out_flight["departure_date"]).days
        if min_duration <= trip_duration <= max_duration:
            trip_start = out_flight["departure_date"]
            trip_end = in_flight["departure_date"]
            overlap_start = max(trip_start, window_start)
            overlap_end = min(trip_end, window_end)
            overlap_days = (overlap_end - overlap_start).days + 1 if overlap_end >= overlap_start else 0
            if overlap_days >= 2:
                total_price = out_flight["price"] + in_flight["price"]
                if total_price < price_threshold:
                    combined_results.append({
                        "outbound": out_flight,
                        "inbound": in_flight,
                        "trip_duration": trip_duration,
                        "total_price": total_price
                    })

# -------------------------
# Print combined results
# -------------------------
print("\n✈️  Combined One-Way Ticket Pairs (Total Price under $700):\n")
if combined_results:
    combined_results.sort(key=lambda x: x["total_price"])
    for idx, combo in enumerate(combined_results, 1):
        out_flight = combo["outbound"]
        in_flight = combo["inbound"]
        print(f"🔹 Option {idx}:")
        print("   Outbound (MOW -> OSA):")
        print(f"       * Departure Date: {out_flight['departure_date']}")
        print(f"       * Price: ${out_flight['price']}")
        print(f"       * Airline: {out_flight['airline']} (Flight No: {out_flight['flight_number']})")
        print(f"       * Departure Time: {out_flight['departure_at']}")
        print(f"       * Ticket Link: {out_flight['link']}")
        print("   Inbound (TYO -> MOW):")
        print(f"       * Departure Date: {in_flight['departure_date']}")
        print(f"       * Price: ${in_flight['price']}")
        print(f"       * Airline: {in_flight['airline']} (Flight No: {in_flight['flight_number']})")
        print(f"       * Departure Time: {in_flight['departure_at']}")
        print(f"       * Ticket Link: {in_flight['link']}")
        print(f"   Trip Duration: {combo['trip_duration']} days")
        print(f"   Total Price: ${combo['total_price']}\n")
else:
    print("No flight combinations found that meet the criteria.")
